# 4. Relaciones entre variables

Punto 5 del enunciado: comparaciones, gráficos y medidas de relación para responder a las preguntas.

**Responsables:** Alex (P1 y P2) y Vahab (P3 y comparación de fuentes).

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent / 'src'))
from config import RAW_ADZUNA, RAW_TECNOEMPLEO, PROCESSED, REFERENCE

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

In [ ]:
# De momento trabajamos con los datos limpios de Tecnoempleo.
# Cuando esté la unión con Jooble, se cambiará por los archivos unificados.
ofertas = pd.read_csv(PROCESSED / 'tecnoempleo_ofertas.csv',
                      parse_dates=['fecha_publicacion', 'fecha_obtencion'])
tecnologias = pd.read_csv(PROCESSED / 'tecnoempleo_tecnologias.csv')
con_salario = ofertas[ofertas['tiene_salario']]
print(ofertas.shape, tecnologias.shape, '· con salario:', len(con_salario))

**Tamaño mínimo de los grupos.** Solo el 30 % de las ofertas publica salario, así que muchos grupos tienen pocos salarios. Comparamos medianas solo en grupos con **al menos 10 salarios** y mostramos siempre el número de ofertas (`n`). Con menos de 10, una sola oferta puede mover mucho la mediana. Si con los datos de Jooble los grupos crecen, se subirá a 20.

In [ ]:
MIN_N = 10

# Estilo común de los gráficos (el mismo que en el cuaderno 03)
AZUL, NARANJA, GRIS = '#2a78d6', '#eb6834', '#52514e'
plt.rcParams.update({
    'figure.figsize': (9, 4), 'figure.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': GRIS, 'axes.titleweight': 'bold',
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.8,
    'xtick.color': GRIS, 'ytick.color': GRIS, 'font.size': 10,
})
from matplotlib.colors import LinearSegmentedColormap
eur = plt.FuncFormatter(lambda x, _: f'{x/1000:,.0f}k €'.replace(',', '.'))

## P1. ¿Qué tecnologías se asocian a salarios más altos? *(Alex)*

Una oferta puede pedir varias tecnologías, así que cuenta en cada una de ellas. Miramos dos cosas: **cuántas ofertas** pide cada tecnología (demanda) y **qué salario** anuncian.

In [ ]:
por_tec = tecnologias.merge(ofertas, on='id_oferta')
tabla_tec = por_tec.groupby('tecnologia').agg(
    ofertas=('id_oferta', 'count'),
    con_salario=('tiene_salario', 'sum'),
    pct_con_salario=('tiene_salario', 'mean'),
    mediana=('salario_medio', 'median'),
    q1=('salario_medio', lambda s: s.quantile(.25)),
    q3=('salario_medio', lambda s: s.quantile(.75)),
).sort_values('ofertas', ascending=False)
tabla_tec.style.format({'pct_con_salario': '{:.0%}', 'mediana': '{:,.0f}', 'q1': '{:,.0f}', 'q3': '{:,.0f}'}, na_rep='—')

### Demanda: ofertas que piden cada tecnología

In [ ]:
top = tabla_tec.head(20).sort_values('ofertas')
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top.index, top['ofertas'], color=AZUL, height=0.65)
for y, v in enumerate(top['ofertas']):
    ax.text(v + 1.5, y, str(v), va='center', color=GRIS, fontsize=9)
ax.set(title='Las 20 tecnologías más pedidas en Tecnoempleo', xlabel='Ofertas')
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

### Salario por tecnología

Mediana del salario medio (punto) y rango intercuartílico (línea: el 50 % central de las ofertas). La línea discontinua es la mediana de todas las ofertas con salario.

In [ ]:
comparables = tabla_tec[tabla_tec['con_salario'] >= MIN_N].sort_values('mediana')
mediana_global = con_salario['salario_medio'].median()

fig, ax = plt.subplots(figsize=(9, 0.5 * len(comparables) + 1.5))
y = np.arange(len(comparables))
ax.hlines(y, comparables['q1'], comparables['q3'], color='#86b6ef', linewidth=4)
ax.plot(comparables['mediana'], y, 'o', color=AZUL, markersize=9, markeredgecolor='white', markeredgewidth=1.5)
ax.axvline(mediana_global, color=GRIS, linestyle='--', linewidth=1)
ax.set_yticks(y, [f'{t}  (n={n})' for t, n in zip(comparables.index, comparables['con_salario'])])
ax.xaxis.set_major_formatter(eur)
ax.set(title=f'Salario anunciado por tecnología (tecnologías con ≥ {MIN_N} salarios)', xlabel='€ brutos al año')
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

pocas = tabla_tec[(tabla_tec['con_salario'] > 0) & (tabla_tec['con_salario'] < MIN_N)]
print(f'Con menos de {MIN_N} salarios (no se comparan): {", ".join(f"{t} ({n})" for t, n in pocas["con_salario"].items())}')
print(f'Sin ningún salario: {", ".join(tabla_tec[tabla_tec["con_salario"] == 0].index)}')

### ¿Las diferencias son significativas?

Usamos la prueba de Kruskal-Wallis, que compara varias distribuciones sin suponer que sean normales (el salario no lo es, como vimos en el EDA univariante).

In [ ]:
from scipy.stats import kruskal
grupos = [g['salario_medio'].dropna() for t, g in por_tec.groupby('tecnologia') if t in comparables.index]
h, p = kruskal(*grupos)
print(f'Kruskal-Wallis: H = {h:.1f} · p = {p:.4f}')

# Excel queda muy por debajo del resto: ¿la diferencia depende solo de él?
grupos_sin_excel = [g['salario_medio'].dropna() for t, g in por_tec.groupby('tecnologia')
                    if t in comparables.index and t != 'Excel']
h2, p2 = kruskal(*grupos_sin_excel)
print(f'Sin Excel:      H = {h2:.1f} · p = {p2:.4f}')

### Conclusiones de P1

- **Las tecnologías más pedidas son Java (141 ofertas), SQL (115), SAP (93) y Python (91)**, seguidas de las nubes AWS y Azure. Las del curso (Google Colab, Vercel y Supabase) no aparecen en ninguna oferta.
- **Solo 10 tecnologías tienen al menos 10 salarios publicados**, así que la comparación salarial se limita a ellas. Del resto no podemos sacar conclusiones sobre salario.
- **Las mejor pagadas son SAP y .NET** (mediana de 49.500 € en ambas), por delante de JavaScript, AWS y Azure (42.000–42.750 €). Java, Linux y Python quedan en torno a la mediana general (37.500 €), y SQL algo por debajo (33.750 €).
- **Excel es claramente la peor pagada** (21.750 €). No es tanto por la herramienta como por el tipo de puesto: aparece sobre todo en ofertas de dependiente, técnico de reparación o auxiliar administrativo.
- **Más demanda no significa más salario:** Java, SQL y Python son de las más pedidas, pero no de las mejor pagadas.
- **Las diferencias son estadísticamente significativas** (Kruskal-Wallis, p < 0,001), y lo siguen siendo sin Excel (p = 0,024).
- **Limitación:** los grupos son pequeños (de 10 a 30 salarios) y los rangos intercuartílicos se solapan mucho, por lo que el orden exacto entre tecnologías cercanas no es fiable. La unión con Jooble debería ampliar los grupos.

## P2. ¿Madrid o Barcelona? *(Alex)*

Solo cuentan las ofertas con ciudad Madrid o Barcelona. Las 100 % remotas no tienen ciudad y se dejan fuera de esta comparación (se estudian en P3).

In [ ]:
mb = ofertas[ofertas['ciudad'].isin(['Madrid', 'Barcelona'])].copy()
resumen_ciudad = mb.groupby('ciudad').agg(
    ofertas=('id_oferta', 'count'),
    con_salario=('tiene_salario', 'sum'),
    pct_con_salario=('tiene_salario', 'mean'),
    media=('salario_medio', 'mean'),
    mediana=('salario_medio', 'median'),
    q1=('salario_medio', lambda s: s.quantile(.25)),
    q3=('salario_medio', lambda s: s.quantile(.75)),
)
resumen_ciudad.style.format({'pct_con_salario': '{:.0%}', 'media': '{:,.0f}', 'mediana': '{:,.0f}', 'q1': '{:,.0f}', 'q3': '{:,.0f}'})

### Qué tecnologías se piden en cada ciudad

Mapa de calor de las 15 tecnologías más pedidas según dónde está la oferta. Aquí sí incluimos las ofertas 100 % remotas como una columna más. Cada casilla muestra el número de ofertas y, entre paréntesis, qué porcentaje de las ofertas de esa tecnología corresponde a esa ubicación (cada fila suma 100 %). El color sigue ese porcentaje.

In [ ]:
ubicaciones = ['Madrid', 'Barcelona', 'Sin ciudad (remoto)']
por_tec_ciudad = tecnologias.merge(ofertas[['id_oferta', 'ciudad']], on='id_oferta')
por_tec_ciudad = por_tec_ciudad[por_tec_ciudad['ciudad'].isin(ubicaciones)]

conteo = pd.crosstab(por_tec_ciudad['tecnologia'], por_tec_ciudad['ciudad'])[ubicaciones]
conteo = conteo.loc[conteo.sum(axis=1).sort_values(ascending=False).index].head(15)
porcentaje = conteo.div(conteo.sum(axis=1), axis=0) * 100
etiquetas = conteo.astype(str) + '\n(' + porcentaje.round(0).astype(int).astype(str) + ' %)'

# Referencia: reparto de TODAS las ofertas entre las tres ubicaciones
reparto_total = ofertas.loc[ofertas['ciudad'].isin(ubicaciones), 'ciudad'].value_counts(normalize=True)[ubicaciones] * 100

secuencial = LinearSegmentedColormap.from_list('seq', ['#f4f8fd', '#86b6ef', AZUL, '#104281'])
fig, ax = plt.subplots(figsize=(8, 9))
sns.heatmap(porcentaje, cmap=secuencial, vmin=0, vmax=100, annot=etiquetas, fmt='',
            annot_kws={'size': 9}, linewidths=2, linecolor='white',
            cbar_kws={'shrink': 0.6, 'label': '% de las ofertas de la tecnología'}, ax=ax)
ax.set_xticklabels([f'{u}\n(total: {reparto_total[u]:.0f} %)' for u in ubicaciones], rotation=0)
ax.set(title='Dónde se piden las 15 tecnologías más demandadas', xlabel='', ylabel='')
ax.grid(False)
plt.tight_layout(); plt.show()

**Cómo leerlo:** el "total" de cada columna es el reparto de todas las ofertas (Madrid 53 %, remoto 25 %, Barcelona 21 %). Una tecnología con un porcentaje claramente mayor que el total en una columna se pide más de lo normal en esa ubicación.

- **Madrid** concentra más de lo normal las ofertas de **Linux (73 %)**, **Excel (72 %)** y **JavaScript (65 %)**.
- **En remoto** se piden más de lo normal **SAP (50 %)**, **Kubernetes (50 %)**, **Node.js (48 %)** y **AWS (47 %)**: tecnologías de cloud, DevOps y consultoría, que se pueden trabajar a distancia.
- **Barcelona** tiene más peso del normal en **Node.js y TypeScript (35 %)** y **React (28 %)**, es decir, en desarrollo web.
- **Excel no aparece en ninguna oferta remota**: va ligado a puestos presenciales de tienda, reparación o administración.

### Salario en cada ciudad

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
datos = [mb.loc[mb['ciudad'] == c, 'salario_medio'].dropna() for c in ['Barcelona', 'Madrid']]
ax.boxplot(datos, vert=False, widths=0.55, patch_artist=True,
           tick_labels=[f'Barcelona (n={len(datos[0])})', f'Madrid (n={len(datos[1])})'],
           boxprops=dict(facecolor='#cde2fb', color=AZUL), medianprops=dict(color=AZUL, linewidth=2),
           whiskerprops=dict(color=AZUL), capprops=dict(color=AZUL),
           flierprops=dict(marker='o', markersize=4, markerfacecolor=AZUL, markeredgecolor='white'))
ax.xaxis.set_major_formatter(eur)
ax.set(title='Salario anunciado: Madrid frente a Barcelona', xlabel='€ brutos al año')
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

from scipy.stats import mannwhitneyu
u, p = mannwhitneyu(datos[1], datos[0])
print(f'Mann-Whitney (Madrid vs Barcelona): U = {u:.0f} · p = {p:.3f}')

### ¿Quién publica los salarios atípicos de Madrid?

Los puntos sueltos de la derecha en la caja de Madrid son salarios muy por encima del resto. Los miramos uno a uno, con la empresa que publica la oferta.

In [ ]:
madrid = mb[(mb['ciudad'] == 'Madrid') & mb['tiene_salario']]
q1, q3 = madrid['salario_medio'].quantile([.25, .75])
limite = q3 + 1.5 * (q3 - q1)
atipicos_madrid = madrid[madrid['salario_medio'] > limite].sort_values('salario_medio', ascending=False)
print(f"Límite superior en Madrid: {limite:,.0f} € · atípicos: {len(atipicos_madrid)}".replace(',', '.'))
print('Empresas que los publican:', atipicos_madrid['empresa'].value_counts().to_dict())
atipicos_madrid[['titulo', 'empresa', 'categoria', 'salario_texto', 'etiquetas_originales']]

Son puestos de dirección, arquitectura o SAP muy especializado (CTO, Head of AI Engineering, arquitectos, *leads*), y **10 de los 13 los publica Michael Page**, una consultora de selección de perfiles senior. Es decir, los salarios más altos de Madrid dependen en buena parte de un solo intermediario: lo tenemos en cuenta como limitación.

### ¿Es una diferencia de ciudad o de tipo de puesto? (efecto composición)

Si una ciudad tiene más ofertas de puestos peor pagados, su salario global sale más bajo aunque pague lo mismo por cada puesto. Miramos qué peso tienen los puestos de helpdesk y soporte, los peor pagados, en cada ciudad.

In [ ]:
RE_SOPORTE = r'helpdesk|soporte|teleoperador'
mb['soporte'] = mb['titulo'].str.contains(RE_SOPORTE, case=False, na=False) | mb['categoria'].isin(['Helpdesk', 'Soporte Técnico'])
mb_sal = mb[mb['tiene_salario']]
pd.DataFrame({
    '% de ofertas de soporte (con salario)': mb_sal.groupby('ciudad')['soporte'].mean(),
    'mediana sin soporte': mb_sal[~mb_sal['soporte']].groupby('ciudad')['salario_medio'].median(),
    'n sin soporte': mb_sal[~mb_sal['soporte']].groupby('ciudad').size(),
}).style.format({'% de ofertas de soporte (con salario)': '{:.0%}', 'mediana sin soporte': '{:,.0f}'})

Comparamos también la mediana **dentro de cada categoría de puesto**, en las categorías con al menos 5 salarios en cada ciudad:

In [ ]:
cat = mb_sal.groupby(['categoria', 'ciudad'])['salario_medio'].agg(['count', 'median']).unstack()
cat = cat[(cat[('count', 'Madrid')] >= 5) & (cat[('count', 'Barcelona')] >= 5)].sort_values(('median', 'Madrid'))

fig, ax = plt.subplots(figsize=(9, 0.55 * len(cat) + 1.5))
y = np.arange(len(cat))
ax.hlines(y, cat[('median', 'Madrid')], cat[('median', 'Barcelona')], color='#c3c2b7', linewidth=2)
ax.plot(cat[('median', 'Madrid')], y, 'o', color=AZUL, markersize=9, markeredgecolor='white', markeredgewidth=1.5, label='Madrid')
ax.plot(cat[('median', 'Barcelona')], y, 'o', color=NARANJA, markersize=9, markeredgecolor='white', markeredgewidth=1.5, label='Barcelona')
ax.set_yticks(y, [f'{c}  (n={int(m)} / {int(b)})' for c, m, b in zip(cat.index, cat[('count', 'Madrid')], cat[('count', 'Barcelona')])])
ax.xaxis.set_major_formatter(eur)
ax.set(title='Mediana del salario por categoría de puesto', xlabel='€ brutos al año (n = Madrid / Barcelona)')
ax.legend(loc='lower right', frameon=False)
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

### Comprobación: solo ofertas recientes y sin jornada parcial

In [ ]:
filtro = mb_sal[mb_sal['reciente'] & ~mb_sal['posible_jornada_parcial']]
filtro.groupby('ciudad')['salario_medio'].agg(['count', 'median', 'mean']).round(0)

### Conclusiones de P2

- **Volumen:** Madrid concentra muchas más ofertas que Barcelona (659 frente a 267, unas 2,5 veces más). Barcelona publica el salario algo más a menudo (34 % frente a 29 %).
- **Qué se pide en cada ciudad:** Madrid concentra más de lo normal Linux, Excel y JavaScript; Barcelona tiene más peso en desarrollo web (Node.js, TypeScript, React); y las tecnologías de cloud, DevOps y SAP se ofrecen mucho en remoto (en torno al 50 % de sus ofertas).
- **Salario global:** Madrid queda ligeramente por encima (mediana de 37.500 € frente a 36.000 €; media de 40.140 € frente a 35.995 €), pero **la diferencia no es estadísticamente significativa** (Mann-Whitney, p = 0,11). Barcelona tiene una distribución más dispersa, con más salarios bajos.
- **Salarios atípicos de Madrid:** los 13 salarios por encima de 65.250 € son puestos de dirección, arquitectura o SAP muy especializado, y 10 de ellos los publica Michael Page, una consultora de selección de perfiles senior. Los salarios altos de Madrid dependen en buena parte de ese intermediario.
- **Efecto composición:** los puestos de helpdesk y soporte, los peor pagados, pesan el triple en Barcelona (24 % de las ofertas con salario, frente al 8 % en Madrid). Sin ellos, **Barcelona pasa por delante en mediana** (42.000 € frente a 39.000 €), con medias casi iguales.
- **Por categoría de puesto:** en 4 de las 6 categorías comparables (jefe de proyecto, consultor, ingeniero y analista programador), Barcelona anuncia salarios iguales o más altos. Madrid solo queda por encima en helpdesk y soporte técnico.
- **Respuesta:** Madrid ofrece mucho más empleo tecnológico, pero **no paga más por el mismo tipo de puesto**. La diferencia global se debe a la mezcla de puestos, no a la ciudad.
- **Limitación:** las comparaciones por categoría se basan en grupos de solo 5 a 24 salarios por ciudad, así que son orientativas.

## P3. ¿Remoto, híbrido o presencial?

In [ ]:
# TODO (Vahab)

## Comparación entre fuentes y salarios estimados frente a declarados

In [ ]:
# TODO (Vahab)